# TT-34: Embeddings + Cosine Similarity, khớp CV với JD


## Bước 1. Cấu hình và nạp dữ liệu


In [3]:
import sys, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
sys.path.append('../src')
from prepare_data import prepare
from matcher import load_model, encode, score_tfidf, score_full, score_chunk
from evaluate import ndcg_at_k, precision_at_k, synonym_pairs, bias_check

ROOT = Path('..'); REPORTS = ROOT / 'reports'; REPORTS.mkdir(exist_ok=True)
MODEL = 'all-MiniLM-L6-v2'          

if not any((ROOT / 'data/cv_an_danh').glob('*.txt')):
    prepare(ROOT / 'data/raw/Resume.csv', ROOT / 'data/cv_an_danh')

cv_files = sorted((ROOT / 'data/cv_an_danh').glob('*.txt'))
jd_files = sorted((ROOT / 'data/jd').glob('*.txt'))
cv_ids, jd_ids = [f.stem for f in cv_files], [f.stem for f in jd_files]
cvs = [f.read_text(encoding='utf-8') for f in cv_files]
jds = [f.read_text(encoding='utf-8') for f in jd_files]
nganh = [i.split('__')[0] for i in cv_ids]         
print(len(cvs), 'CV,', len(jds), 'JD')

30 CV, 5 JD


## Bước 2. Nhãn vàng (chấm tay 0-3 cho từng cặp CV, JD)


In [7]:
df = pd.read_csv(ROOT / 'data/nhan_vang.csv')
df['nhan'] = df['goi_y']
df.to_csv(ROOT / 'data/nhan_vang.csv', index=False)

In [ ]:
JD_CAT = {'JD_IT': 'INFORMATION-TECHNOLOGY', 'JD_HR': 'HR', 'JD_ACCOUNTANT': 'ACCOUNTANT',
          'JD_SALES': 'SALES', 'JD_TEACHER': 'TEACHER'}

def goi_y(cv_id, jd_id):
    c, j = cv_id.split('__')[0], JD_CAT[jd_id]
    return 3 if c == j else 2 if (c, j) == ('FINANCE', 'ACCOUNTANT') else 0

lab_path = ROOT / 'data/nhan_vang.csv'
if not lab_path.exists():
    pd.DataFrame([(c, j, goi_y(c, j), '') for c in cv_ids for j in jd_ids],
                 columns=['cv_id', 'jd_id', 'goi_y', 'nhan']).to_csv(lab_path, index=False)
    raise FileNotFoundError(f'Đã tạo {lab_path}: hãy chấm tay cột nhan (0-3) rồi chạy lại.')

lab = pd.read_csv(lab_path).pivot(index='jd_id', columns='cv_id', values='nhan').loc[jd_ids, cv_ids]
assert lab.notna().all().all(), 'Còn cặp chưa chấm nhãn'
LABELS = lab.values.astype(int)                     

## Bước 3. Kiểm tra độ dài CV theo token (xác nhận bị cắt cụt)

In [10]:
model = load_model(MODEL)
n_tok = np.array([len(model.tokenizer.tokenize(c)) for c in cvs])
df_len = pd.DataFrame({'cv_id': cv_ids, 'so_token': n_tok})
df_len['ti_le_bi_bo'] = (1 - model.max_seq_length / df_len.so_token).clip(lower=0).round(2)
df_len.to_csv(REPORTS / 'do_dai_cv.csv', index=False)
print('max_seq_length =', model.max_seq_length,
      '| CV bị cắt:', (n_tok > model.max_seq_length).mean().round(2),
      '| tỉ lệ nội dung bị bỏ trung bình:', df_len.ti_le_bi_bo.mean().round(2))

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1222 > 256). Running this sequence through the model will result in indexing errors


max_seq_length = 256 | CV bị cắt: 1.0 | tỉ lệ nội dung bị bỏ trung bình: 0.76


## Bước 4-6. Ba cách xếp hạng: TF-IDF, embedding nguyên CV, embedding + chunking

In [11]:
S = {'TF-IDF': score_tfidf(jds, cvs),
     'Embedding (nguyên CV)': score_full(model, jds, cvs),
     'Embedding + chunking': score_chunk(model, jds, cvs)}

## Bước 7. So sánh bằng NDCG@10 và Precision@10 (đối chiếu nhãn vàng)

In [12]:
res = pd.DataFrame({n: {'NDCG@10': ndcg_at_k(s, LABELS), 'P@10': precision_at_k(s, LABELS)}
                    for n, s in S.items()}).T.round(3)
res.to_csv(REPORTS / 'so_sanh_3_cach.csv')
ax = res.plot.bar(rot=0, figsize=(8, 4), ylim=(0, 1), title='So sánh 3 cách xếp hạng CV')
for c in ax.containers: ax.bar_label(c, fmt='%.2f')
plt.tight_layout(); plt.savefig(REPORTS / 'so_sanh_3_cach.png', dpi=150); plt.close()
print(res)

                       NDCG@10  P@10
TF-IDF                   0.882  0.52
Embedding (nguyên CV)    0.933  0.56
Embedding + chunking     0.942  0.56


## Bước 8 và 12. So sánh model embedding, đo tốc độ encode và tra cứu

In [13]:
MODELS = ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'paraphrase-multilingual-MiniLM-L12-v2']
rows = []
for name in MODELS:
    m = load_model(name)
    t = time.time(); s = score_chunk(m, jds, cvs); t_all = time.time() - t     # encode toàn bộ CV + chấm điểm
    V = encode(m, cvs)                                                         # kho vector dựng sẵn để đo tra cứu
    t = time.time(); q = encode(m, [jds[0]]); np.argsort(-(q @ V.T)[0]); t_q = time.time() - t
    rows.append({'model': name, 'NDCG@10': ndcg_at_k(s, LABELS), 'P@10': precision_at_k(s, LABELS),
                 'chieu': V.shape[1], 'max_seq': m.max_seq_length,
                 'dung_luong_MB': sum(p.numel() for p in m.parameters()) * 4 / 2**20,
                 'encode_1000_CV_giay(uoc_tinh)': t_all / len(cvs) * 1000,
                 'tra_cuu_1_JD_ms': t_q * 1000})
cmp_models = pd.DataFrame(rows).round(3)
cmp_models.to_csv(REPORTS / 'so_sanh_model.csv', index=False)
print(cmp_models)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ADMIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

                                   model  NDCG@10  P@10  chieu  max_seq  \
0                       all-MiniLM-L6-v2    0.942  0.56    384      256   
1                      all-mpnet-base-v2    0.906  0.58    768      384   
2  paraphrase-multilingual-MiniLM-L12-v2    0.929  0.56    384      128   

   dung_luong_MB  encode_1000_CV_giay(uoc_tinh)  tra_cuu_1_JD_ms  
0         86.644                        147.779           46.209  
1        417.658                       1727.777          153.424  
2        448.813                       1263.775          547.817  


## Bước 9. Mười cặp đồng nghĩa: embedding khớp, TF-IDF bỏ sót
Danh sách ứng viên có thể thêm bớt; hàm chỉ giữ cặp thoả (embedding >= 0.5 và TF-IDF <= 0.1).

In [14]:
PAIRS = [('software developer', 'programmer'), ('front-end framework', 'React'), ('ReactJS', 'React'),
         ('database', 'SQL'), ('machine learning', 'ML'), ('HR', 'human resources'),
         ('recruitment', 'talent acquisition'), ('employee relations', 'staff conflict resolution'),
         ('payroll', 'compensation administration'), ('bookkeeping', 'accounting'),
         ('accounts payable', 'vendor invoices'), ('financial statements', 'balance sheet'),
         ('sales representative', 'account executive'), ('lead generation', 'prospecting'),
         ('customer service', 'client support'), ('lesson plans', 'curriculum'),
         ('classroom management', 'student behavior'), ('teacher', 'educator'),
         ('project management', 'program coordination'), ('cloud services', 'AWS')]
syn = synonym_pairs(lambda t: encode(model, t), PAIRS, jds + cvs)
lines = ['# Cặp đồng nghĩa mà TF-IDF bỏ sót', '', '| Cụm A | Cụm B | Cosine embedding | Cosine TF-IDF |', '|---|---|---|---|']
lines += [f'| {r.a} | {r.b} | {r.embedding:.2f} | {r.tfidf:.2f} |' for r in syn.itertuples()]
(REPORTS / 'cap_dong_nghia.md').write_text('\n'.join(lines), encoding='utf-8')
print(len(syn), 'cặp thoả điều kiện (cần >= 10, nếu thiếu hãy thêm cặp vào PAIRS)')

10 cặp thoả điều kiện (cần >= 10, nếu thiếu hãy thêm cặp vào PAIRS)


## Bước 10. Trực quan hoá vector CV (PCA và t-SNE, tô màu theo ngành)

In [15]:
V = encode(model, cvs)
proj = {'PCA': PCA(2).fit_transform(V),
        't-SNE': TSNE(2, perplexity=min(10, len(cvs) - 1), random_state=0).fit_transform(V)}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (title, X) in zip(axes, proj.items()):
    for g in sorted(set(nganh)):
        mask = np.array(nganh) == g
        ax.scatter(X[mask, 0], X[mask, 1], label=g, s=40)
    ax.set_title(title)
axes[0].legend(title='Ngành', fontsize=7)
plt.tight_layout(); plt.savefig(REPORTS / 'tsne_cv.png', dpi=150); plt.close()

## Bước 11. Kiểm tra thiên lệch giới tính
CV Kaggle không có tên: ta gắn dòng 'Name' (tên nam, rồi tên nữ cùng họ) vào đầu cùng một CV, giữ nguyên nội dung, rồi so điểm và thứ hạng.

In [16]:
NAME_PAIRS = [('John Miller', 'Emily Miller'), ('Michael Brown', 'Sarah Brown'),
              ('David Wilson', 'Jessica Wilson'), ('Robert Taylor', 'Jennifer Taylor')]
fns = {'embedding_nguyen_cv': lambda j, c: score_full(model, j, c),
       'embedding_chunking': lambda j, c: score_chunk(model, j, c)}
per_cv, summ = [], []
for name, fn in fns.items():
    d, s = bias_check(fn, jds, cvs, NAME_PAIRS)
    d.insert(0, 'cv_id', cv_ids); d.insert(1, 'method', name); per_cv.append(d)
    summ.append({'method': name, **s})
pd.concat(per_cv).to_csv(REPORTS / 'bias_check.csv', index=False)
bias_sum = pd.DataFrame(summ).round(4)
bias_sum.to_csv(REPORTS / 'bias_check_tong_hop.csv', index=False)
print(bias_sum.T)

                                                 0                   1
method                         embedding_nguyen_cv  embedding_chunking
chenh_lech_tb                               0.0016              0.0005
chenh_lech_tuyet_doi_tb                     0.0137              0.0025
so_voi_do_lech_chuan_cua_diem               0.0975              0.0224
doi_hang_tb                                   0.92              0.2933
p_wilcoxon                                  0.4569              0.0335


## Tổng kết file đầu ra

In [17]:
for f in sorted(REPORTS.iterdir()): print(f.name)

bias_check.csv
bias_check_tong_hop.csv
cap_dong_nghia.md
do_dai_cv.csv
so_sanh_3_cach.csv
so_sanh_3_cach.png
so_sanh_model.csv
tsne_cv.png
